# Laboratorio 7 (1-2)
## Grupo 1

### Importación de librerías.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

### Importación de datos.

In [2]:
df = pd.read_csv('datos_limpios.csv')
df.head(2)

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,...,room_type_Hotel room,room_type_Private room,room_type_Shared room,calendar_last_scraped_2025-09-16,calendar_last_scraped_2025-09-17,calendar_last_scraped_2025-09-22,calendar_last_scraped_2025-09-23,calendar_last_scraped_2025-09-24,calendar_last_scraped_2025-09-25,calendar_last_scraped_2025-09-30
0,1.0,2.0,30.26057,-97.73441,3,1.0,1.0,2.0,97.0,2,...,0,0,0,0,1,0,0,0,0,0
1,1.0,2.0,30.26034,-97.76487,2,1.0,1.0,2.0,160.0,3,...,0,0,0,0,1,0,0,0,0,0


### Estadísticas del objetivo.

In [3]:
df['price'].describe()

count    73549.000000
mean       769.542958
std       4326.270197
min          8.000000
25%        120.000000
50%        194.000000
75%        330.000000
max      50123.000000
Name: price, dtype: float64

### Manejo de NaN

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 73549 entries, 0 to 164211
Data columns (total 67 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_listings_count                           73549 non-null  float64
 1   host_total_listings_count                     73549 non-null  float64
 2   latitude                                      73549 non-null  float64
 3   longitude                                     73549 non-null  float64
 4   accommodates                                  73549 non-null  int64  
 5   bathrooms                                     73549 non-null  float64
 6   bedrooms                                      73549 non-null  float64
 7   beds                                          73549 non-null  float64
 8   price                                         73549 non-null  float64
 9   minimum_nights                                73549 non-null  int

### Se reemplazará los NaN por las medianas.

In [5]:
df = df.fillna(df.median(numeric_only=True))

Las categorías se harán a partir de los cuartiles. Aquellas que estén debajo del cuartil 1 se denominarán 'Económicas', las que estén entre el primer y el tercer cuartil se denominarán 'Intermedia' y el resto 'Cara'.

### Se crea la categoría y se elimina la variable price.

In [6]:
categoria = []
for x in df['price']:
    if x <= 120:
        categoria.append('Económica')
    elif x < 330:
        categoria.append('Intermedia')
    else:
        categoria.append('Cara')
df = df.drop(columns=['price'])
df['categoria'] = categoria

### Volviendo númerica las categorías de precio. (Inciso 1)

In [7]:
dummies = pd.get_dummies(df['categoria'], dtype=int)
df = pd.concat([df, dummies], axis=1)
df = df.drop(columns=['categoria'])
df.head(2)

,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,...,calendar_last_scraped_2025-09-16,calendar_last_scraped_2025-09-17,calendar_last_scraped_2025-09-22,calendar_last_scraped_2025-09-23,calendar_last_scraped_2025-09-24,calendar_last_scraped_2025-09-25,calendar_last_scraped_2025-09-30,Cara,Económica,Intermedia
0,1.0,2.0,30.26057,-97.73441,3,1.0,1.0,2.0,2,90,...,0,1,0,0,0,0,0,0,1,0
1,1.0,2.0,30.26034,-97.76487,2,1.0,1.0,2.0,3,365,...,0,1,0,0,0,0,0,0,0,1


### Separación de datos. (Inciso 2)

In [8]:
y = df[['Cara', 'Económica', 'Intermedia']]
df = df.drop(columns=['Cara', 'Económica', 'Intermedia'])
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=0, stratify=categoria)

### Escalado de datos.

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
y_train_mc = y_train[['Cara', 'Económica', 'Intermedia']].idxmax(axis=1)
y_test_mc  = y_test[['Cara', 'Económica', 'Intermedia']].idxmax(axis=1)

### Modelo.

In [20]:
log_reg = LogisticRegression(
    solver='lbfgs',
    class_weight = 'balanced',
    random_state = 0,
    max_iter = 1000
)

### GridSearch.

In [21]:
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2']}
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1 
)

### Entrenamiento del modelo y predicción.

In [22]:
grid_search.fit(X_train_scaled, y_train_mc)
y_pred = grid_search.predict(X_test_scaled)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


c:\Users\ricro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


### Evaluación del modelo.

In [23]:
# Mejor modelo.
modelo = grid_search.best_estimator_

# Predicción
y_pred = modelo.predict(X_test_scaled)
y_prob = modelo.predict_proba(X_test_scaled)

# Evaluación
print("Accuracy:", accuracy_score(y_test_mc, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test_mc, y_pred))

cm = confusion_matrix(y_test_mc, y_pred, labels=modelo.classes_)
cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in modelo.classes_],
    columns=[f"Pred_{c}" for c in modelo.classes_]
)

print("\nMatriz de confusión:\n")
print(cm_df)

Accuracy: 0.8766825288919102

Classification report:

              precision    recall  f1-score   support

        Cara       0.80      0.92      0.86      3688
   Económica       0.87      0.93      0.90      3704
  Intermedia       0.93      0.83      0.88      7318

    accuracy                           0.88     14710
   macro avg       0.87      0.89      0.88     14710
weighted avg       0.88      0.88      0.88     14710


Matriz de confusión:

                 Pred_Cara  Pred_Económica  Pred_Intermedia
Real_Cara             3375              81              232
Real_Económica          47            3434              223
Real_Intermedia        780             451             6087


### Conclusiones